# StyleTTS2 Fine-tuning Guide

!!! NOTEBOOK STILL HAS LOTS OF WORK TO DO. I'M FOCUSING FIRST ON getting "Trelis_StyleTTS2_Finetune_Demo.ipynb" working well.

> This is a private/commercial notebook built by Trelis Research. Purchase a copy at Trelis.com.

This Jupyter notebook guides you through the process of fine-tuning a StyleTTS2 model for text-to-speech tasks. It covers setting up the environment, preparing the data, configuring the model, and initiating the training process.

Subscribe to Trelis Research emails here and get notified each time a new video tutorial is published.

Attribution:
This notebook builds on scripts from the following repos:
- [StyleTTS2](https://github.com/yl4579/StyleTTS2).

## Choosing a GPU

You have a few options:
1. Hosted service (Recommended), e.g. Runpod:
- Runpod one-click template [here](https://runpod.io/gsc?template=ifyqsvjlzj&ref=jmfkcdio) - easier setup.
2. Google Colab (fine for 7B models or smaller):
- Upload the .ipynb notebook
- Select a T4 GPU from Runtime -> Change Runtime Type.
3. Your own computer (assuming you have an AMD or Nvidia GPU) - ADVANCED:
- Set up jupyter lab and a virtual environment using the instructions [here](https://github.com/TrelisResearch/install-guides/blob/main/jupyter-lab-setup.md)

## Environment Setup
First, we set up the necessary environment by installing required packages and tools.


In [4]:
!python -m pip install --upgrade pip
!apt-get update > /dev/null # update quietly
!apt-get install git-lfs
!pip install pandas -q
!pip install -q SoundFile torchaudio munch torch pydub pyyaml librosa nltk matplotlib accelerate transformers phonemizer einops einops-exts tqdm typing-extensions git+https://github.com/resemble-ai/monotonic_align.git -qU
!pip install huggingface_hub hf_transfer -qU
!apt-get install espeak-ng -y 
!pip install -q tensorboard peft==0.6.0

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 89 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 32.2 MB/s eta 0:00:00a 0:00:01
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 89 not upgraded.


In [ ]:
# Turn on fast Rust downloads/uploads
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## Preparing the Dataset directory

Here, we prepare the data directory and download the necessary voice dataset.
This downloads the voice dataset from Hugging Face, moves it to the Data directory, and removes the temporary download folder. This dataset is specifically curated by Trelis Research for optimal fine-tuning results. You can generate your own dataset with the `dataset_curation.ipynb` script in the ADVANCED-transcription repo.

In [3]:
# Clone the original StyleTTS2 repo - we'll need these scripts for fine-tuning
!git clone https://github.com/yl4579/StyleTTS2.git
%cd StyleTTS2

Cloning into 'StyleTTS2'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (245/245), done.5% (111/245)
remote: Compressing objects: 100% (102/102), done.
remote: Total 372 (delta 201), reused 143 (delta 143), pack-reused 127
Receiving objects: 100% (372/372), 133.96 MiB | 22.42 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/workspace/StyleTTS2


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
from huggingface_hub import hf_hub_download, HfApi, Repository
import shutil
import os

# Define the model repository and file paths
model_repo = "yl4579/StyleTTS2-LibriTTS"
model_path = "Models/LibriTTS"
target_files = ["config.yml", "epochs_2nd_00020.pth"]
local_dir = "Models"

# Create a local directory to store the downloaded files
os.makedirs(local_dir, exist_ok=True)

# Download the files from the HuggingFace model repository
downloaded_files = []
for file_name in target_files:
    file_path = hf_hub_download(repo_id=model_repo, filename=f"{model_path}/{file_name}", local_dir=local_dir)
    downloaded_files.append(file_path)

Models/LibriTTS/config.yml:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

epochs_2nd_00020.pth:   0%|          | 0.00/771M [00:00<?, ?B/s]

In [ ]:
# Download the TTS model that we will be fine-tuning

!git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LibriTTS
!mv StyleTTS2-LibriTTS/Models .

In [14]:
!rm -rf Data/train_list.txt 
!rm -rf Data/val_list.txt 
!rm -rf Data/wavs

In [15]:
!git-lfs clone https://huggingface.co/datasets/Trelis/trelis_voice 
!mv ./trelis_voice/* ./Data/ 
!rm -rf trelis_voice

          with new flags from 'git clone'

'git clone' has been updated in upstream Git to have comparable
speeds to 'git lfs clone'.
Cloning into 'trelis_voice'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 71 (delta 0), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (71/71), 30.26 KiB | 108.00 KiB/s, done.


## Configuring the Model

Now we configure the model parameters according to Trelis Research recommendations.

This loads the Trelis-specific configuration file, updates key parameters (data path, batch size, max length, and joint epoch), and saves the updated configuration. These settings are optimized for the Trelis voice dataset and typical GPU configurations. Try and experiment different configurations according to your computational resources.

In [16]:
config_path = "Configs/config_ft.yml"

import yaml
config = yaml.safe_load(open(config_path))

config['data_params']['root_path'] = "Data/wavs" # write the path to wavs directory 
config['batch_size'] = 4 # choose according to your RAM
config['max_len'] = 700 # Choose according to your RAM
config['loss_params']['joint_epoch'] = 110 # don't do SLM Adverserial training if you don't have enough ram

with open(config_path, 'w') as outfile:
  yaml.dump(config, outfile, default_flow_style=True)

In [17]:
!python train_lora_finetune.py --config_path ./Configs/config_ft.yml --lora_r 16 --lora_alpha 32 

python: can't open file '/workspace/StyleTTS2/train_lora_finetune.py': [Errno 2] No such file or directory


## Push the Trained Model To Hub

In [ ]:
!pip install huggingface_hub

from huggingface_hub import login, create_repo, upload_folder

# Log in to Hugging Face
login()

# Create a new model repository
repo_name = "your-username/your-model-name"  # Replace with your desired repository name
create_repo(repo_name, repo_type="model")

# Upload checkpoints
upload_folder(
    folder_path="./checkpoints",  # Replace with the path to your checkpoints folder
    repo_id=repo_name,
    repo_type="model",
    path_in_repo="checkpoints"
)

# Upload config files
upload_folder(
    folder_path="./config",  # Replace with the path to your config files folder
    repo_id=repo_name,
    repo_type="model",
    path_in_repo="config"
)

print(f"Checkpoints and config files uploaded successfully to {repo_name}")

## Conclusion
This Trelis Research notebook sets up the environment for fine-tuning a StyleTTS2 model using the Trelis voice dataset and initiates the training process. However, the error at the end indicates that some debugging or adjustments may be necessary before successful training can be achieved. Make sure to review the data preparation steps and model configuration if you encounter similar issues.